In [37]:
import yfinance as yf
import pandas as pd
import talib
import torch
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [2]:
# data = yf.download("^GDAXI", start="2019-01-01", end="2024-01-01")
# data.to_csv('index_stock.csv')

df = pd.read_csv('index_stock.csv')
print(df.shape)
print(df.head())


(1274, 6)
        Price             Close              High               Low  \
0      Ticker            ^GDAXI            ^GDAXI            ^GDAXI   
1        Date               NaN               NaN               NaN   
2  2019-01-02  10580.1904296875  10612.7197265625  10386.9697265625   
3  2019-01-03    10416.66015625    10538.66015625  10400.1103515625   
4  2019-01-04  10767.6904296875    10786.33984375   10483.900390625   

               Open    Volume  
0            ^GDAXI    ^GDAXI  
1               NaN       NaN  
2    10477.76953125  79626700  
3  10467.1103515625  84733800  
4  10533.9404296875  95339500  


In [3]:
df = df.iloc[2:]

In [4]:
df

,Price,Close,High,Low,Open,Volume
2,2019-01-02,10580.1904296875,10612.7197265625,10386.9697265625,10477.76953125,79626700
3,2019-01-03,10416.66015625,10538.66015625,10400.1103515625,10467.1103515625,84733800
4,2019-01-04,10767.6904296875,10786.33984375,10483.900390625,10533.9404296875,95339500
5,2019-01-07,10747.8095703125,10814.4697265625,10681.26953125,10814.3896484375,71151400
6,2019-01-08,10803.98046875,10910.7099609375,10745.0302734375,10750.1904296875,93672200
...,...,...,...,...,...,...
1269,2023-12-21,16687.419921875,16708.349609375,16624.16015625,16667.310546875,57871300
1270,2023-12-22,16706.1796875,16735.3203125,16651.779296875,16673.30078125,46295300
1271,2023-12-27,16742.0703125,16775.7109375,16697.580078125,16727.76953125,37678900
1272,2023-12-28,16701.55078125,16783.7890625,16688.51953125,16780.94921875,36091600


In [5]:
df['Doji'] = talib.CDLDOJI(df['Open'], df['High'], df['Low'], df['Close'])
df['Hammer'] = talib.CDLHAMMER(df['Open'], df['High'], df['Low'], df['Close'])
df['Engulfing'] = talib.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])


In [6]:
def min_max_normalization(x, columns=[]):
    x = x.astype(float)
    x_scaled = (x - x.min()) / (x.max() - x.min())
    return x_scaled


In [7]:
df[['Close', 'High', 'Low', 'Open', 'Volume']] = df[['Close', 'High', 'Low', 'Open', 'Volume']].apply(min_max_normalization)

In [8]:
df = df.rename(columns = {'Price':'Date'})

In [9]:
df = df.reset_index()

In [10]:
df.drop(columns='index')
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0,0,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0,0,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0,0,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0,0,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0,0,0


In [11]:
df.value_counts(['Doji', 'Hammer', 'Engulfing'])

Doji  Hammer  Engulfing
0     0        0           997
100   0        0           177
0     0       -100          45
               100          28
      100      0            22
100   100      0             3
Name: count, dtype: int64

In [12]:
def define_pattern(x):
    if x['Doji'] == 0 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 0
    elif x['Doji'] == 100 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 1
    elif x['Doji'] == 0 and x['Hammer'] == 100 and x['Engulfing'] == 0:
        return 2
    elif x['Doji'] == 0 and x['Hammer'] == 0 and (x['Engulfing'] != 0):
        return 3
    else:
        return -1 
    


In [13]:
df['Pattern'] = df.apply(define_pattern, axis=1)

In [14]:
df

,index,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0,0,0,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0,0,0,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0,0,0,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0,0,0,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
1267,1269,2023-12-21,0.987189,0.964615,0.984013,0.966700,0.146646,0,0,0,0
1268,1270,2023-12-22,0.989435,0.967851,0.987261,0.967409,0.116946,0,0,0,0
1269,1271,2023-12-27,0.993731,0.972697,0.992646,0.973853,0.094839,0,0,0,0
1270,1272,2023-12-28,0.988880,0.973666,0.991581,0.980144,0.090766,0,0,-100,3


In [15]:
df['Pattern'].value_counts()

Pattern
 0    997
 1    177
 3     73
 2     22
-1      3
Name: count, dtype: int64

In [16]:
df = df[df['Pattern'] != -1]
df = df.drop(columns =['Doji', 'Hammer', 'Engulfing'])

In [17]:
df.head(20)

,index,Date,Close,High,Low,Open,Volume,Pattern
0,2,2019-01-02,0.256022,0.233268,0.250612,0.234457,0.202463,0
1,3,2019-01-03,0.236444,0.224382,0.252157,0.233196,0.215566,0
2,4,2019-01-04,0.278470,0.254098,0.262009,0.241102,0.242776,0
3,5,2019-01-07,0.276090,0.257473,0.285217,0.274280,0.180718,0
4,6,2019-01-08,0.282815,0.269020,0.292714,0.266685,0.238499,0
5,7,2019-01-09,0.293510,0.275164,0.303774,0.282604,0.244656,0
6,8,2019-01-10,0.296895,0.270990,0.297773,0.277770,0.196379,0
7,9,2019-01-11,0.292809,0.274814,0.302696,0.290220,0.191970,0
8,10,2019-01-14,0.289032,0.266083,0.297620,0.275044,0.167362,0
9,11,2019-01-15,0.293327,0.279113,0.300658,0.291855,0.203927,0


## Building neural network with Torch

In [18]:
W1 = torch.randn(5,8,requires_grad = True)
b1 = torch.zeros(8,requires_grad = True)
W2 = torch.randn(8,4,requires_grad = True)
b2 = torch.zeros(4,requires_grad = True)

In [19]:
def forward_pass(X):
    hidden_lay_pre_act = torch.matmul(X, W1) + b1
    relu_act = torch.relu(hidden_lay_pre_act)
    logits = torch.matmul(relu_act, W2) + b2
    return logits

In [20]:
freq = df['Pattern'].value_counts().sort_index()
weights = 1/freq

In [21]:
tensor_1 = torch.tensor(weights.values, dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight = tensor_1)

In [22]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)

In [23]:
for epoch in range(100):
    logits = forward_pass(torch.tensor(df[['Close', 'High', 'Low', 'Open', 'Volume']].values, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(df['Pattern'].values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

2.6609649658203125
1.4658430814743042
1.4954074621200562
1.4032630920410156
1.4067370891571045
1.3974308967590332
1.3914986848831177
1.3896435499191284
1.3873494863510132
1.3855475187301636


## Doing using normal train-test split

In [24]:
df_train, df_test = train_test_split(df, test_size = 0.2)

In [25]:
X_train = df_train[['Close', 'High', 'Low', 'Open', 'Volume']]
X_test = df_test[['Close', 'High', 'Low', 'Open', 'Volume']]
y_train = df_train['Pattern']
y_test = df_test['Pattern']


In [62]:
scaler_normal = StandardScaler()
X_train_scaled = scaler_normal.fit_transform(X_train)
X_test_scaled = scaler_normal.transform(X_test)

C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [64]:
for epoch in range(500):
    logits = forward_pass(torch.tensor(X_train_scaled, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(y_train.values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

3.463592290878296
1.7050155401229858
1.464310646057129
1.224200963973999
1.1861945390701294
1.1530654430389404
1.1310820579528809
1.111796498298645
1.0982952117919922
1.084376335144043
1.063931941986084
1.021252989768982
0.9564508199691772
0.9328832626342773
0.9166063070297241
0.9002622961997986
0.8757705092430115
0.8369350433349609
0.7921990156173706
0.7825494408607483
0.7744858264923096
0.7689324617385864
0.7650658488273621
0.7621983289718628
0.7597389221191406
0.7576577067375183
0.7557293176651001
0.754112958908081
0.7527351975440979
0.7514523863792419
0.750315248966217
0.7489912509918213
0.7476073503494263
0.7463384866714478
0.7451093792915344
0.743894636631012
0.7427088618278503
0.7415710687637329
0.7404452562332153
0.7394227981567383
0.738469660282135
0.7375551462173462
0.7367182970046997
0.735926628112793
0.7351577877998352
0.73441481590271
0.7336918115615845
0.7330063581466675
0.7323492169380188
0.7317057251930237


In [66]:
logits_test = forward_pass(torch.tensor(X_test, dtype=torch.float32))
logits_test

tensor([[  1.6603, -12.4745,   0.9858,   1.7862],
        [ -1.1799,  -0.3949,  -3.4226,  -1.8670],
        [ -0.8282,  -0.2499,  -3.8188,  -1.5594],
        ...,
        [  1.4167,  -0.2247,  -5.1693,   0.9614],
        [  2.4624,  -8.8197,  -3.8078,   2.9825],
        [  9.6948, -31.9344,   0.6425,  11.1271]], grad_fn=<AddBackward0>)

In [67]:
test_result = torch.argmax(logits_test, dim=1)
test_result

tensor([3, 1, 1, 0, 2, 0, 1, 0, 0, 0, 0, 3, 1, 0, 1, 2, 0, 1, 3, 0, 0, 3, 3, 3,
        1, 0, 0, 3, 1, 1, 1, 0, 3, 0, 1, 1, 1, 0, 2, 0, 0, 1, 0, 0, 1, 0, 3, 3,
        1, 3, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 2, 3, 3, 0, 1, 0, 3, 3, 1, 0, 0, 3,
        0, 1, 0, 1, 3, 1, 0, 0, 0, 1, 3, 1, 1, 1, 2, 3, 0, 0, 0, 0, 2, 0, 1, 0,
        1, 0, 0, 3, 0, 1, 3, 2, 0, 0, 0, 3, 1, 0, 1, 2, 0, 3, 1, 0, 2, 0, 3, 3,
        0, 1, 3, 1, 3, 1, 0, 1, 0, 1, 1, 0, 0, 2, 3, 0, 0, 1, 0, 0, 0, 1, 1, 2,
        1, 0, 1, 1, 1, 1, 0, 0, 3, 1, 3, 3, 1, 1, 1, 2, 0, 0, 0, 0, 3, 0, 1, 1,
        1, 1, 1, 2, 1, 0, 3, 0, 1, 1, 3, 2, 0, 0, 3, 1, 0, 0, 0, 0, 1, 3, 1, 0,
        1, 1, 3, 1, 1, 3, 3, 3, 0, 1, 3, 1, 0, 1, 0, 3, 0, 0, 1, 3, 3, 0, 1, 1,
        1, 3, 2, 1, 2, 0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 1, 1, 0, 0, 0, 0, 0, 1, 1,
        1, 0, 1, 1, 3, 1, 0, 1, 2, 1, 0, 0, 3, 3])

In [68]:
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.94      0.46      0.62       195
           1       0.30      0.87      0.44        31
           2       0.21      0.40      0.28        10
           3       0.25      0.67      0.36        18

    accuracy                           0.52       254
   macro avg       0.42      0.60      0.43       254
weighted avg       0.78      0.52      0.57       254



In [30]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [31]:
for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

1.3683979511260986
1.2354280948638916
1.3413268327713013
1.187791347503662
1.4376529455184937
1.2756632566452026
1.5361380577087402
1.3462456464767456
1.5045526027679443
1.3405340909957886
1.4318156242370605
1.5218842029571533
1.264201283454895
1.6812466382980347
1.4126776456832886
1.3182190656661987
1.2217377424240112
1.598706603050232
1.1765296459197998
1.2182198762893677
1.2267177104949951
1.1418014764785767
1.1102255582809448
1.308485746383667
1.1208410263061523
1.1444456577301025
2.3586888313293457
1.9335418939590454
1.1118903160095215
1.2206823825836182
1.2584141492843628
1.2579829692840576
1.4994057416915894
1.2152514457702637
1.3778671026229858
1.512357473373413
1.0858149528503418
1.2391334772109985
1.187421202659607
1.1717884540557861
1.2209758758544922
1.417731523513794
1.435203194618225
1.124928593635559
1.1459952592849731
1.2412678003311157
1.3177968263626099
1.5665514469146729
1.4605919122695923
1.281632661819458
1.3232190608978271
1.2374179363250732
1.512174367904663
1.31

In [32]:
logits_test_batch = forward_pass(torch.tensor(X_test.values, dtype=torch.float32))
test_result = torch.argmax(logits_test_batch, dim=1)
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.77      0.27      0.40       195
           1       0.14      0.61      0.23        31
           2       0.00      0.00      0.00        10
           3       0.07      0.17      0.10        18

    accuracy                           0.30       254
   macro avg       0.24      0.26      0.18       254
weighted avg       0.61      0.30      0.34       254



In [69]:
y_train.value_counts()

Pattern
0    802
1    146
3     55
2     12
Name: count, dtype: int64

In [ ]:
sm = SMOTE(sampling_strategy={1:400, 2:400, 3:400})
X_train_partial_sm, y_train_partial_sm = 

## Applying imbalanced learn

In [33]:
sm = SMOTE()
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

In [36]:
y_train_sm.value_counts()

Pattern
3    802
0    802
1    802
2    802
Name: count, dtype: int64

In [51]:
scaler = StandardScaler()
scaler_1 = scaler.fit(X_train)
X_train_sm = scaler_1.transform(X_train_sm)
X_test_sm = scaler_1.transform(X_test)

C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [52]:
X_train_sm

array([[-7.75081514, -7.91561079, -7.72754202, -7.95364686, -2.7357407 ],
       [-3.02871979, -2.86096908, -3.01902505, -2.73275032, 23.47615795],
       [-7.3642248 , -7.57359788, -7.33458034, -7.54779337, -5.01459387],
       ...,
       [-6.42249139, -6.29779174, -6.43242183, -6.15723151, -1.52892429],
       [ 1.04930922,  1.16305193,  1.09800945,  1.24767119, -8.80415984],
       [ 2.82987141,  2.79674964,  2.63657154,  2.73010178, -3.59885657]],
      shape=(3208, 5))

In [53]:
X_train_sm = torch.tensor(X_train_sm, dtype=torch.float32)
y_train_sm = torch.tensor(y_train_sm, dtype=torch.long)

dataset = TensorDataset(X_train_sm, y_train_sm)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

C:\Users\USER\AppData\Local\Temp\ipykernel_40268\2942018516.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_sm = torch.tensor(y_train_sm, dtype=torch.long)


In [54]:
 for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

100.02243041992188
90.01861572265625
87.29467010498047
20.76238441467285
9.085517883300781
55.10868453979492
65.02256774902344
94.97692108154297
26.068044662475586
18.887218475341797
22.610946655273438
14.75804328918457
26.334909439086914
41.625282287597656
11.44057846069336
6.331490516662598
10.396130561828613
26.401823043823242
32.797176361083984
4.529830455780029
20.17009735107422
12.093696594238281
6.470258712768555
11.074352264404297
17.841598510742188
13.091679573059082
12.782931327819824
7.6161298751831055
4.1518449783325195
23.18633270263672
9.911158561706543
19.60047149658203
1.5992647409439087
5.369505405426025
4.7104902267456055
6.402195453643799
6.185781955718994
2.139009714126587
2.840399980545044
6.142848014831543
4.836019039154053
10.545116424560547
5.3059892654418945
9.269389152526855
3.7641329765319824
1.4361767768859863
2.164360284805298
0.9086177945137024
4.345366477966309
4.354528903961182
5.329564094543457
2.763134717941284
5.77982759475708
6.448581695556641
2.6284

In [58]:
logits_test_batch_sm = forward_pass(torch.tensor(X_test_sm,dtype=torch.float32))
test_result_sm = torch.argmax(logits_test_batch_sm, dim=1)
print(classification_report(y_test, test_result_sm.numpy()))

              precision    recall  f1-score   support

           0       1.00      0.01      0.02       195
           1       0.30      0.77      0.43        31
           2       0.21      0.30      0.25        10
           3       0.10      0.89      0.18        18

    accuracy                           0.18       254
   macro avg       0.40      0.49      0.22       254
weighted avg       0.82      0.18      0.09       254



## Using partial smote for class 1,2,3

In [72]:
sm = SMOTE(sampling_strategy={1:400, 2:400, 3:400})
X_train_partial_sm, y_train_partial_sm= sm.fit_resample(X_train_scaled, y_train)

In [77]:
X_train_partial_sm_tensor = torch.tensor(X_train_partial_sm, dtype=torch.float32)
y_train_partial_sm_tensor = torch.tensor(y_train_partial_sm, dtype=torch.long)

dataset_partial_sm = TensorDataset(X_train_partial_sm_tensor, y_train_partial_sm_tensor)
loader_partial_sm = DataLoader(dataset, batch_size=32, shuffle=True)

In [78]:
 for epoch in range(500):
    for X_batch, y_batch in loader_partial_sm:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

0.5365168452262878
0.5667194724082947
0.372213751077652
0.398631751537323
0.3448191285133362
0.32930341362953186
0.37598511576652527
0.3089423179626465
0.5038843154907227
0.514828622341156
0.3306044638156891
0.3332614600658417
0.30531612038612366
0.29381510615348816
0.3962377607822418
0.28689271211624146
0.5282073616981506
0.26010456681251526
0.18885383009910583
0.2573309540748596
0.19917571544647217
0.20778335630893707
0.3206399381160736
0.4233512580394745
0.22318045794963837
0.47875508666038513
0.29191264510154724
0.38428670167922974
0.3808315396308899
0.26172342896461487
0.2684502601623535
0.33084097504615784
0.40421977639198303
0.26417815685272217
0.2827073335647583
0.3649449646472931
0.27155542373657227
0.4881562292575836
0.24293842911720276
0.3636889159679413
0.4397585093975067
0.2531244456768036
0.1994735449552536
0.45326802134513855
0.44378724694252014
0.3358626663684845
0.32757478952407837
0.40212705731391907
0.3083493709564209
0.6181771159172058
0.4320354759693146
0.240326151

In [80]:
logits_test_batch_partial_sm = forward_pass(torch.tensor(X_test_scaled,dtype=torch.float32))
test_result_partial_sm = torch.argmax(logits_test_batch_partial_sm, dim=1)
print(classification_report(y_test, test_result_partial_sm.numpy()))

              precision    recall  f1-score   support

           0       0.84      0.16      0.27       195
           1       0.24      0.42      0.31        31
           2       0.25      0.10      0.14        10
           3       0.08      0.72      0.15        18

    accuracy                           0.23       254
   macro avg       0.35      0.35      0.22       254
weighted avg       0.69      0.23      0.26       254



In [ ]:
print(classification_report(y_test, ))